In [2]:
import torch

In [3]:
from utils.fix_probs import masking4 as masking

In [4]:
from utils.logsumexp import logsumexp_infsafe as logsumexp

In [5]:
batch_size, num_vertices, vertex_lens, device = 2, 5, torch.Tensor([3, 4]), torch.device('cpu')

In [6]:
m, r = masking(batch_size, num_vertices, vertex_lens, device)

In [7]:
m

tensor([[[False,  True,  True, False, False],
         [False, False,  True, False, False],
         [ True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True]],

        [[False,  True,  True,  True, False],
         [False, False,  True,  True, False],
         [False, False, False,  True, False],
         [ True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True]]])

In [8]:
r

tensor([[[False],
         [False],
         [ True],
         [ True],
         [ True]],

        [[False],
         [False],
         [False],
         [ True],
         [ True]]])

In [9]:
num_heads, chunk_size = 2, 3

In [10]:
test_q = torch.rand(batch_size, num_vertices, num_heads, chunk_size)
test_k = torch.rand(batch_size, num_vertices, num_heads, chunk_size)

In [11]:
attn = torch.einsum("bicf,bjcf->bijc", test_q, test_k) / (chunk_size ** 0.5)

In [25]:
attn.shape

torch.Size([2, 5, 5, 2])

In [12]:
attn_m = attn.masked_fill(~m.unsqueeze(-1), float('-inf'))

In [13]:
attn_m_log = torch.log_softmax(attn_m, dim=2)

In [14]:
attn_m_log[0][:, :, 0]

tensor([[   -inf, -0.6327, -0.7574,    -inf,    -inf],
        [   -inf,    -inf,  0.0000,    -inf,    -inf],
        [-1.4550, -1.5897, -1.7991, -1.5373, -1.7027],
        [-1.4994, -1.5747, -1.8007, -1.5625, -1.6354],
        [-1.5576, -1.5673, -1.6914, -1.6056, -1.6311]])

In [15]:
attn_m_log_r = attn_m_log.masked_fill(r.unsqueeze(-1), float('-inf'))

In [16]:
attn_m_log_r[0][:, :, 0]

tensor([[   -inf, -0.6327, -0.7574,    -inf,    -inf],
        [   -inf,    -inf,  0.0000,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf]])

In [23]:
attn_m_log_r[0][:, :, 1]

tensor([[   -inf, -0.5769, -0.8248,    -inf,    -inf],
        [   -inf,    -inf,  0.0000,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf]])

In [17]:
transition_matrix = logsumexp(attn_m_log_r, dim=-1)

In [18]:
transition_matrix.shape

torch.Size([2, 5, 5, 1])

In [19]:
transition_matrix = transition_matrix.squeeze(-1)

In [20]:
transition_matrix[0]

tensor([[   -inf,  0.0887, -0.0974,    -inf,    -inf],
        [   -inf,    -inf,  0.6931,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf]])

In [21]:
transition_matrix[1]

tensor([[   -inf, -0.3512, -0.3985, -0.4703,    -inf],
        [   -inf,    -inf, -0.0336,  0.0325,    -inf],
        [   -inf,    -inf,    -inf,  0.6931,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf]])

In [28]:
torch.exp(transition_matrix[0])

tensor([[0.0000, 1.0928, 0.9072, 0.0000, 0.0000],
        [0.0000, 0.0000, 2.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [29]:
torch.exp(transition_matrix[1])

tensor([[0.0000, 0.7039, 0.6713, 0.6248, 0.0000],
        [0.0000, 0.0000, 0.9670, 1.0330, 0.0000],
        [0.0000, 0.0000, 0.0000, 2.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [34]:
vocab_size = 4 # 0 is pad

In [35]:
vocab_log_probs = torch.rand(batch_size, num_vertices, vocab_size)
vocab_log_probs = torch.log_softmax(vocab_log_probs, dim=-1)

In [41]:
targets = torch.tensor([[1, 2, 3, 0, 0], [1, 2, 3, 2, 0]])

In [55]:
for i, nex_prob in enumerate(transition_matrix[0][1]):
    print(nex_prob)

tensor(-inf)
tensor(-inf)
tensor(0.6931)
tensor(-inf)
tensor(-inf)


In [59]:
targets[0]

tensor([1, 2, 3, 0, 0])

In [107]:
def dfs(trans, vocab, target, max_len, length, curr_index, ending_index, curr_score):
    if length > max_len:
        return float('-inf')
    
    if curr_index == ending_index:
        return curr_score if length == max_len else float('-inf')
    
    paths = []
    for next_index, next_prob in enumerate(trans[curr_index]):
        if next_prob != float('-inf'):
            next_score = vocab[next_index][target[length]]
            path_score = dfs(trans, vocab, target, max_len, length + 1, next_index, ending_index, next_score)
            if path_score != float('-inf'):
                paths.append(path_score + next_prob + curr_score)
    if len(paths) == 0:
        return float('-inf')
    else:
        paths = torch.stack(paths)
        paths = paths.flatten()
        return logsumexp(torch.tensor(paths), dim=0).squeeze()

In [109]:
res = dfs(transition_matrix[0], vocab_log_probs[0], targets[0], 3, 1, 0, 2, vocab_log_probs[0][targets[0][0]])

C:\Users\John\AppData\Local\Temp\ipykernel_7812\3163833908.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return logsumexp(torch.tensor(paths), dim=0).squeeze()


In [110]:
torch.exp(res)

tensor(0.1036)

In [62]:
transition_matrix[0]

tensor([[   -inf,  0.0887, -0.0974,    -inf,    -inf],
        [   -inf,    -inf,  0.6931,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf]])

In [63]:
vocab_log_probs[0]

tensor([[-1.3245, -1.2217, -1.2043, -1.9701],
        [-1.3849, -1.4280, -1.3264, -1.4089],
        [-1.3176, -1.0387, -1.6111, -1.7224],
        [-1.8322, -1.5849, -1.5236, -0.8746],
        [-1.5886, -1.7036, -0.8296, -1.7285]])

In [124]:
targets[1]

tensor([1, 2, 3, 2, 0])

In [122]:
torch.exp(transition_matrix[1])

tensor([[0.0000, 0.7039, 0.6713, 0.6248, 0.0000],
        [0.0000, 0.0000, 0.9670, 1.0330, 0.0000],
        [0.0000, 0.0000, 0.0000, 2.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [123]:
torch.exp(vocab_log_probs[1])

tensor([[0.2106, 0.2029, 0.3317, 0.2548],
        [0.2428, 0.1475, 0.2974, 0.3122],
        [0.2943, 0.2417, 0.2388, 0.2252],
        [0.2971, 0.2986, 0.2290, 0.1753],
        [0.2831, 0.3148, 0.2252, 0.1768]])

In [112]:
1.09 * 2 * 0.2947 * 0.2654 * 0.1786

0.03045222307624001

In [125]:
0.7039 * 0.967 * 2 * 0.2029 * 0.2974 * 0.2252 * 0.2290

0.004236374202126179

In [128]:
test = torch.log(torch.tensor([1.09 * 2 * 0.2947 * 0.2654 * 0.1786, 0.7039 * 0.967 * 2 * 0.2029 * 0.2974 * 0.2252 * 0.2290]))

In [129]:
test

tensor([-3.4916, -5.4640])

In [130]:
test / torch.tensor([3, 4])

tensor([-1.1639, -1.3660])

In [131]:
-torch.mean(test / torch.tensor([3, 4]))

tensor(1.2649)

In [113]:
torch.exp(res)

tensor(0.1036)

In [114]:
vertex_lens = torch.Tensor([3, 4])
target_lens = torch.Tensor([3, 4])

In [118]:
targets = targets.long()
vertex_lens = vertex_lens.long()
target_lens = target_lens.long()

In [115]:
from losses.dag_loss import dag_loss

In [148]:
result = dag_loss(targets, transition_matrix, vocab_log_probs, target_lens, vertex_lens)

In [138]:
test / torch.tensor([3, 4])

tensor([-1.1639, -1.3660])

In [139]:
torch.mean(test / torch.tensor([3, 4]))

tensor(-1.2649)

In [149]:
result

tensor(1.2644)

In [144]:
result

tensor(1.2644)

In [147]:
import torch
from utils.vector_gather import vector_gather
from utils.logsumexp import logsumexp_infsafe as logsumexp

def dag_loss_raw(targets, transition_matrix, emission_probs):
    """
    Calculates the directed acyclic graph (DAG) loss given the targets, transition matrix, and emission probabilities.
    It returns the dynamic programming table of which one of the entries is the DAG loss.

    Args:
        targets (torch.Tensor): The target sequence of shape (batch_size, m).
        transition_matrix (torch.Tensor): The transition matrix of shape (batch_size, l, l).
        emission_probs (torch.Tensor): The emission probabilities of shape (batch_size, l, vocab_size).

    Returns:
        torch.Tensor: The DAG loss of shape (batch_size, m, l).
    """
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.ones((batch_size, m, l))
    dp[dp == 1] = -float('inf')
    initial_probs = torch.gather(emission_probs, dim=2, index=targets[:, 0].unsqueeze(1).unsqueeze(2))
    dp[:, 0, 0] = initial_probs.squeeze(2).squeeze(1)
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    dp = dp.to(transition_matrix.device)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + ((logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1)).squeeze(1))
    return dp

def process_dp(dp, target_lens, vertex_lens):
    """
    Processes the dynamic programming table (dp) to extract the correct loss values.
    The target lengths and vertex lengths are needed to determine which values to extract
    and which values are a result of padding and should be ignored.

    Args:
        dp (torch.Tensor): The dynamic programming table of shape (batch_size, m, l).
        target_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the length of each target sequence.
        vertex_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the number of non-padding vertices for each batch.

    Returns:
        torch.Tensor: The values corresponding to the last target and last vertex of shape (batch_size,).
    """
    dp_values = vector_gather(dp, target_lens - 1)
    values = torch.gather(dp_values, dim=1, index=(vertex_lens - 1).unsqueeze(-1))
    return values

def dag_loss(targets, transition_matrix, emission_probs, target_lens, vertex_lens):
    """
    Calculates the directed acyclic graph (DAG) loss given the targets, transition matrix, and emission probabilities.

    Args:
        targets (torch.Tensor): The target sequence of shape (batch_size, m).
        transition_matrix (torch.Tensor): The transition matrix of shape (batch_size, l, l).
        emission_probs (torch.Tensor): The emission probabilities of shape (batch_size, l, vocab_size).
        target_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the length of each target sequence.
        vertex_lens (torch.Tensor): A tensor of shape (batch_size,) that describes the number of non-padding vertices for each batch.

    Returns:
        torch.Tensor: The DAG loss of shape (batch_size,).
    """
    dp = dag_loss_raw(targets, transition_matrix, emission_probs)
    values = process_dp(dp, target_lens, vertex_lens)
    values = values.flatten() / target_lens
    return -torch.mean(values)